In [3]:
import os
import pandas as pd
import networkx as nx
from pathlib import Path

In [4]:
data_dir = Path("data")

# Change these once you confirm the actual filenames
edges_path = data_dir / "large_twitch_edges.csv"
nodes_path = data_dir / "large_twitch_features.csv"   # or features.csv / nodes.csv, depending on release

edges = pd.read_csv(edges_path)
nodes = pd.read_csv(nodes_path)

rename columns as described in the [Paper](https://arxiv.org/abs/2101.03091)

In [5]:
edges = edges.rename(columns={"numeric_id_1": "src_id", "numeric_id_2": "target_id"})
nodes = nodes.rename(columns={
    "views": "view_count",
    "mature": "explicit_content",
    "life_time": "account_lifetime",
    "created_at": "creation_date",
    "updated_at": "last_stream",
    "numeric_id": "id",
    "affiliate": "affiliate_status"
})
# order the cols
nodes = nodes[
    ["id", "dead_account", "language", "affiliate_status", "explicit_content", "creation_date", "last_stream", "view_count", "account_lifetime"]
]

In [7]:
save = input("Save dataframes? y/n: ")
if "y" in save:
    nodes.to_csv(data_dir / "renamed_large_twitch_edges.csv")
    edges.to_csv(data_dir / "renamed_large_twitch_features.csv")

Save dataframes? y/n:  y


In [8]:
edges.head()

,src_id,target_id
0,98343,141493
1,98343,58736
2,98343,140703
3,98343,151401
4,98343,157118


In [9]:
nodes.head()

,id,dead_account,language,affiliate_status,explicit_content,creation_date,last_stream,view_count,account_lifetime
0,0,0,EN,1,1,2016-02-16,2018-10-12,7879,969
1,1,0,EN,0,0,2011-05-19,2018-10-08,500,2699
2,2,0,EN,1,1,2010-02-27,2018-10-12,382502,3149
3,3,0,EN,0,0,2015-01-26,2018-10-01,386,1344
4,4,0,EN,0,0,2013-11-22,2018-10-11,2486,1784


In [10]:
print(edges.shape, nodes.shape)

(6797557, 2) (168114, 9)


In [11]:
G = nx.from_pandas_edgelist(edges, source="src_id", target="target_id")
print(G.number_of_nodes(), G.number_of_edges())

168114 6797557


In [7]:
# deg = dict(G.degree())
# clustering = nx.clustering(G)
# pagerank = nx.pagerank(G, alpha=0.85)

# feat_df = pd.DataFrame({
#     "node_id": list(G.nodes()),
#     "degree": [deg[n] for n in G.nodes()],
#     "clustering": [clustering[n] for n in G.nodes()],
#     "pagerank": [pagerank[n] for n in G.nodes()],
# })
nodes = list(G.nodes())
deg = dict(G.degree())

feat_df = pd.DataFrame({
    "node_id": nodes,
    "degree": [deg[n] for n in nodes],
})

In [8]:
core = nx.core_number(G)   # can still take time, but often manageable
feat_df["core_number"] = [core[n] for n in nodes]

KeyboardInterrupt: 